GROUP NUMBER: 3

GROUP MEMBER: A.HEMANT KUMAAR, JACK LICHWA , PRITHIKA K


# **PIT STOP ANALYSIS PROJECT**

## **PROJECT BACKGROUND**

Formula 1 (F1) is the pinnacle of motorsport, combining elite driver skill with cutting-edge engineering. A critical and often race-deciding element of F1 strategy is the **pit stop**  a brief pause in which a team changes tires, makes aerodynamic adjustments, or responds to race incidents. The duration of a pit stop, even down to fractions of a second, can determine whether a driver retains or loses a podium position.

For top-finishing drivers (those who finish in the top 3), pit stop strategy is especially high-stakes. A slow stop can cost a driver a podium; an efficiently timed stop can vault them ahead of a competitor. Understanding what factors influence the **maximum pit stop duration** experienced by top 3 finishers has implications for race engineering, team strategy optimization, and performance analytics.

This study investigates whether historical race data, tire information, and race conditions can be used to predict the maximum pit stop duration among the top three drivers in a Formula 1 race. Can the maximum pit stop duration for the top 3 drivers in a Formula 1 race be predicted using race context and historical performance data?

## **DATASET**

**Source**: All datasets come from Kaggle: “Formula 1 Race Data” by jtrotman. https://www.kaggle.com/datasets/jtrotman/formula-1-race-data?resource=download&select=lap_times.csv

They contain historical Formula 1 data from 1950 onwards, including race results, circuits, drivers, constructors, lap times, pit stops, qualifying, sprint results, and season standings.

**Key Variable**:
At this stage, we are exploring all variables from the pit stops, races, circuits, drivers, constructors, lap times, qualifying, and results datasets. Final selection of key predictor variables will be based on further analysis and exploratory data analysis (EDA), including correlations, distributions, and relevance to pit stop duration.

**Data Type**: The datasets include a mix of numeric, categorical, ordinal, and datetime variables:

- Numeric: milliseconds, lap, round, year, alt, points, positionOrder
- Categorical: driverId, surname, constructorId, constructor name, circuitId, location, country, status
- Ordinal: stop (pit stop sequence), grid position, and finishing positions
- Datetime: race dates (races.date) and driver birth dates (drivers.dob)

These variables provide the foundation for preprocessing and exploratory analysis to identify potential predictors for pit stop duration.

## LAB <TEMPLATE>

### **SETUP AND DATA LOADING**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (11, 5)

print('Libraries loaded.')

Libraries loaded.


In [3]:
f1_top3_race_regression_df = pd.read_csv('https://raw.githubusercontent.com/SU-Machine-Learning-II-Group3/PIT-STOP-ANALYSIS/refs/heads/main/data/f1_top3_pitstop_regression_data.csv')
f1_driver_classification_df = pd.read_csv('https://raw.githubusercontent.com/SU-Machine-Learning-II-Group3/PIT-STOP-ANALYSIS/refs/heads/main/data/f1_driver_classification_data.csv')
print('Regression dataset:', f1_top3_race_regression_df.shape)
print('Classification dataset:', f1_driver_classification_df.shape)
display(f1_driver_classification_df.head())
display(f1_top3_race_regression_df.head())

# --- Regression dataset column categories ---
reg_cols = f1_top3_race_regression_df.columns.tolist()
reg_categories = {
    "race_context":    [c for c in reg_cols if c in ["race_year", "race_circuit", "race_country", "race_altitude_m"]],
    "top3_features":   [c for c in reg_cols if c.startswith("top3_")],
    "race_aggregates": [c for c in reg_cols if c.startswith("race_")],
    "driver_features": [c for c in reg_cols if c.startswith(("pos1_", "pos2_", "pos3_"))],
}
print("=== Regression Dataset Column Categories ===")
for group, cols in reg_categories.items():
    print(f"  {group} ({len(cols)}): {cols}")

# --- Classification dataset column categories ---
clf_cols = f1_driver_classification_df.columns.tolist()
clf_categories = {
    "race_context":    [c for c in clf_cols if c in ["race_year", "race_circuit", "race_country", "race_altitude_m"]],
    "driver_features": [c for c in clf_cols if c in ["team", "race_start_position", "qualifying_position", "quali_lap_time_s", "laps_completed", "points"]],
    "pit_features":    [c for c in clf_cols if "pit" in c],
    "target":          ["top3_finish"],
}
print("=== Classification Dataset Column Categories ===")
for group, cols in clf_categories.items():
    print(f"  {group} ({len(cols)}): {cols}")

display(
    pd.DataFrame(
        [{"dataset": "regression",     "category": k, "n_columns": len(v)} for k, v in reg_categories.items()] +
        [{"dataset": "classification", "category": k, "n_columns": len(v)} for k, v in clf_categories.items()]
    )
)

Regression dataset: (446, 51)
Classification dataset: (5970, 18)


,race_year,race_circuit,race_country,race_altitude_m,team,race_start_position,qualifying_position,quali_lap_time_s,laps_completed,points,driver_pit_stops_count,driver_pit_stops_avg_duration_s,race_pit_stops_count,race_pit_stops_avg_duration_s,race_pit_stops_max_duration_s,race_pit_stops_min_duration_s,race_avg_laps_before_pit_stop,top3_finish
0,2011,Albert Park Grand Prix Circuit,Australia,10,Red Bull,1.0,1.0,85.296,58,25.0,2.0,23.319500,45.0,24.280365,37.856,16.867,23.055556,1
1,2011,Albert Park Grand Prix Circuit,Australia,10,McLaren,2.0,2.0,85.384,58,18.0,2.0,23.213000,45.0,24.280365,37.856,16.867,23.055556,1
2,2011,Albert Park Grand Prix Circuit,Australia,10,Renault,6.0,6.0,85.543,58,15.0,2.0,25.109000,45.0,24.280365,37.856,16.867,23.055556,1
3,2011,Albert Park Grand Prix Circuit,Australia,10,Ferrari,5.0,5.0,85.707,58,12.0,3.0,24.055000,45.0,24.280365,37.856,16.867,23.055556,0
4,2011,Albert Park Grand Prix Circuit,Australia,10,Red Bull,3.0,3.0,85.900,58,10.0,3.0,24.058667,45.0,24.280365,37.856,16.867,23.055556,0


,race_year,race_circuit,race_country,race_altitude_m,top3_pit_stops_max_duration_s,top3_pit_stops_avg_duration_s,top3_pit_stops_min_duration_s,top3_pit_stops_count,top3_avg_laps_before_pit_stop,top3_avg_laps_completed,...,pos3_avg_laps_before_pit_stop,pos3_laps_completed,pos3_pit_stops_avg_duration_s,pos3_pit_stops_count,pos3_pit_stops_max_duration_s,pos3_pit_stops_min_duration_s,pos3_quali_lap_time_s,pos3_qualifying_position,pos3_race_start_position,pos3_team
0,2003,Circuit Gilles Villeneuve,Canada,13,32.485,31.440500,30.229,6.0,33.166667,70.0,...,32.000000,70,31.6265,2.0,32.110,31.143,75.923,2.0,2.0,Williams
1,2003,Nürburgring,Germany,578,33.339,32.821833,32.103,6.0,29.000000,60.0,...,27.000000,60,33.1790,2.0,33.339,33.019,91.780,5.0,5.0,Ferrari
2,2003,Circuit de Nevers Magny-Cours,France,228,23.606,20.948333,19.642,9.0,34.333333,70.0,...,34.666667,70,20.5230,3.0,20.712,20.325,75.480,3.0,3.0,Ferrari
3,2003,Silverstone Circuit,UK,153,41.882,35.278167,32.401,6.0,24.666667,60.0,...,23.500000,60,33.7925,2.0,34.107,33.478,81.695,3.0,3.0,McLaren
4,2003,Hockenheimring,Germany,103,31.207,29.728444,27.638,7.0,29.777778,67.0,...,26.000000,67,30.4770,2.0,31.207,29.747,75.679,4.0,4.0,Renault


=== Regression Dataset Column Categories ===
  race_context (4): ['race_year', 'race_circuit', 'race_country', 'race_altitude_m']
  top3_features (12): ['top3_pit_stops_max_duration_s', 'top3_pit_stops_avg_duration_s', 'top3_pit_stops_min_duration_s', 'top3_pit_stops_count', 'top3_avg_laps_before_pit_stop', 'top3_avg_laps_completed', 'top3_avg_pit_stops_avg_duration_s', 'top3_avg_pit_stops_count', 'top3_avg_pit_stops_max_duration_s', 'top3_avg_pit_stops_min_duration_s', 'top3_avg_quali_lap_time_s', 'top3_avg_race_start_position']
  race_aggregates (9): ['race_year', 'race_circuit', 'race_country', 'race_altitude_m', 'race_pit_stops_max_duration_s', 'race_pit_stops_avg_duration_s', 'race_pit_stops_min_duration_s', 'race_pit_stops_count', 'race_avg_laps_before_pit_stop']
  driver_features (30): ['pos1_avg_laps_before_pit_stop', 'pos1_laps_completed', 'pos1_pit_stops_avg_duration_s', 'pos1_pit_stops_count', 'pos1_pit_stops_max_duration_s', 'pos1_pit_stops_min_duration_s', 'pos1_quali_lap_

,dataset,category,n_columns
0,regression,race_context,4
1,regression,top3_features,12
2,regression,race_aggregates,9
3,regression,driver_features,30
4,classification,race_context,4
5,classification,driver_features,6
6,classification,pit_features,7
7,classification,target,1
